In [1]:
%load_ext autoreload
%autoreload 2

# Openning data

In [2]:
# tests/preprocessing/datasets.ipynb
from pathlib import Path
import sys, os

# point to your project root
project_root = Path(r"/home/galencarmedeiro/git/postdoc/ragtree")
#project_root = Path(r"C:\Users\henri\Documents\git\post-doc\ragtree")
os.chdir(project_root)  # so relative paths go to data/, not tests/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("CWD:", os.getcwd())

CWD: /home/galencarmedeiro/git/postdoc/ragtree


# Docred

In [ ]:
%run "scripts/run_ontology_linking.py" \
  --dataset-key docred_causal \
  --ontology-key docredontology \
  --method llm_embedding \
  --doc-types dev

In [9]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key docred_causal \
  --doc-types train_annotated \
  --skip 0

[kg] wrote: /home/galencarmedeiro/git/postdoc/ragtree/data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json
[kg] stats: {'num_docs': 3053, 'num_triples': 38180, 'num_nodes': 59493, 'num_edges': 38180}


#### Agentic Single Simple RAG

In [ ]:
%run "scripts/run_langgraph_agentic_simple_relations.py" \
  --dataset-key docred_causal \
  --backend vllm \
  --doc-types dev \
  --shot-num 3 \
  --shot-doc-types all \
  --max-llm-calls 2

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method langgraph_agentic_simple \
  --backend vllm \
  --doc-type dev

#### Agent Single Hybrid RAG

In [12]:
%run "scripts/run_agentic_hybrid_relations.py" \
  --dataset-key docred_causal \
  --backend vllm \
  --doc-types dev \
  --ontology-links-path data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl \
  --ontology-key docredontology \
  --ontology-method llm_embedding \
  --kg-path data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json \
  --shot-num 3 \
  --shot-doc-types all \
  --max-llm-calls 1

[agentic_hybrid] Loaded DocRED rel_info with 96 entries.
[agentic_hybrid] input=data/preprocessed/docred_causal.jsonl
[agentic_hybrid] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.agentic_hybrid.vllm.jsonl
[agentic_hybrid] ontology_artifact=data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl
[agentic_hybrid] kg_artifact=data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json
[agentic_hybrid] predict doc-types=['dev'] skip=0 limit=None
[agentic_hybrid] backend=vllm model=openai/gpt-oss-20b
[agentic_hybrid] few-shots collected: 3


AgenticHybrid on docred_causal: 106924doc [5:54:00,  5.03doc/s]   

[agentic_hybrid] done. predicted=998
[agentic_hybrid] wrote: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.agentic_hybrid.vllm.jsonl


In [5]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method agentic_hybrid \
  --backend vllm \
  --doc-type dev

[eval] dataset-key: docred_causal
[eval] method: agentic_hybrid
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: data/preprocessed/docred_causal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.agentic_hybrid.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal/agentic_hybrid.vllm.dev.json

=== Micro-level metrics ===
Precision: 0.3239
Recall:    0.0890
F1:        0.1396

=== Counts ===
TP: 1092
FP: 2279
FN: 11183
num_docs_seen: 998
num_docs_eval: 998
num_docs_missing_gold: 0

[eval] Done.


#### LangGraph Single Agentic RAG

In [17]:
%run "scripts/run_langgraph_agentic_hybrid_relations.py" \
  --dataset-key docred_causal \
  --backend vllm \
  --doc-types dev \
  --ontology-links-path data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl \
  --ontology-key docredontology \
  --ontology-method llm_embedding \
  --kg-path data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json \
  --kg-max-triples 40 \
  --shot-num 0 \
  --planner-mode llm \
  --enable-web \
  --enable-wikidata \
  --max-llm-calls 2

[langgraph_agentic_hybrid] Loaded DocRED rel_info with 96 entries.
[langgraph_agentic_hybrid] input=data/preprocessed/docred_causal.jsonl
[langgraph_agentic_hybrid] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.langgraph_agentic_hybrid.vllm.jsonl
[langgraph_agentic_hybrid] ontology_artifact=data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl
[langgraph_agentic_hybrid] kg_artifact=data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json
[langgraph_agentic_hybrid] predict doc-types=['dev'] skip=0 limit=None
[langgraph_agentic_hybrid] backend=vllm model=openai/gpt-oss-20b
[langgraph_agentic_hybrid] planner_mode=llm max_llm_calls=2 web=True wikidata=True


LangGraphAgenticHybrid on docred_causal: 106924doc [10:11:31,  2.91doc/s]   

[langgraph_agentic_hybrid] done. predicted=998
[langgraph_agentic_hybrid] wrote: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.langgraph_agentic_hybrid.vllm.jsonl


In [18]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method langgraph_agentic_hybrid  \
  --backend vllm \
  --doc-type dev

[eval] dataset-key: docred_causal
[eval] method: langgraph_agentic_hybrid
[eval] backend: vllm
[eval] doc-type: dev
[eval] gold: data/preprocessed/docred_causal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/docred_causal.langgraph_agentic_hybrid.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/docred_causal/langgraph_agentic_hybrid.vllm.dev.json

=== Micro-level metrics ===
Precision: 0.2510
Recall:    0.1799
F1:        0.2096

=== Counts ===
TP: 2208
FP: 6590
FN: 10067
num_docs_seen: 998
num_docs_eval: 998
num_docs_missing_gold: 0

[eval] Done.


#### MARAG

In [ ]:
%run "scripts/run_marag_relations.py" \
  --dataset-key docred_causal \
  --backend vllm \
  --doc-types dev \
  --ontology-key docredontology \
  --ontology-links-path data/preprocessed/docred_causal_olink_llm_embedding_onto_docredontology.jsonl \
  --kg-path data/kg/docred_causal__types=train_annotated_skip=0_limit=None__kg.json \
  --enable-planner \
  --enable-web \
  --enable-wikidata \
  --max-llm-calls 2 \
  --shot-num 3 \
  --shot-type dev

In [ ]:
%run "scripts/eval_relations.py" \
  --dataset-key docred_causal \
  --method marag \
  --backend vllm \
  --doc-type dev

# EventStoryLine

In [ ]:
try:
    %run "scripts/run_ontology_linking.py" --dataset-key eventstoryline --ontology-key owltime --method llm_embedding --doc-types all
except Exception as e:
    print(e)

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key eventstoryline \
  --doc-types full \
  --limit 10

#### Agent Single Hybrid RAG

In [5]:
%run "scripts/run_agentic_hybrid_relations.py" \
  --dataset-key eventstoryline \
  --backend vllm \
  --doc-types all \
  --ontology-links-path data/preprocessed/eventstoryline_olink_llm_embedding_onto_owltime.jsonl \
  --ontology-key owltime \
  --ontology-method llm_embedding \
  --kg-path data/kg/eventstoryline__types=full_skip=0_limit=10__kg.json \
  --shot-num 3 \
  --shot-doc-types all \
  --max-llm-calls 1

[agentic_hybrid] input=data/preprocessed/eventstoryline.jsonl
[agentic_hybrid] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.agentic_hybrid.vllm.jsonl
[agentic_hybrid] ontology_artifact=data/preprocessed/eventstoryline_olink_llm_embedding_onto_owltime.jsonl
[agentic_hybrid] kg_artifact=data/kg/eventstoryline__types=full_skip=0_limit=10__kg.json
[agentic_hybrid] predict doc-types=all skip=0 limit=None
[agentic_hybrid] backend=vllm model=openai/gpt-oss-20b
[agentic_hybrid] few-shots collected: 3


AgenticHybrid on eventstoryline: 443doc [5:00:06, 40.65s/doc]

[agentic_hybrid] done. predicted=443
[agentic_hybrid] wrote: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.agentic_hybrid.vllm.jsonl


In [6]:
%run "scripts/eval_relations.py" \
  --dataset-key eventstoryline \
  --method agentic_hybrid \
  --backend vllm \
  --doc-type all

[eval] dataset-key: eventstoryline
[eval] method: agentic_hybrid
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/eventstoryline.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/eventstoryline.agentic_hybrid.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/eventstoryline/agentic_hybrid.vllm.all.json

=== Micro-level metrics ===
Precision: 0.2086
Recall:    0.0732
F1:        0.1084

=== Counts ===
TP: 706
FP: 2678
FN: 8934
num_docs_seen: 443
num_docs_eval: 443
num_docs_missing_gold: 0

[eval] Done.


# FinCausal

In [ ]:
try:
    %run "scripts/run_ontology_linking.py" --dataset-key fincausal --ontology-key fibocoreplus --method llm_embedding --doc-types all
except Exception as e:
    print(e)

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key fincausal \
  --doc-types train.csv \
  --skip 0

#### Agent Single Hybrid RAG

In [7]:
%run "scripts/run_agentic_hybrid_relations.py" \
  --dataset-key fincausal \
  --backend vllm \
  --doc-types all \
  --ontology-links-path data/preprocessed/fincausal_olink_llm_embedding_onto_fibocoreplus.jsonl \
  --ontology-key fibocoreplus \
  --ontology-method llm_embedding \
  --kg-path data/kg/fincausal__types=train.csv_skip=0_limit=None__kg.json \
  --shot-num 3 \
  --shot-doc-types all \
  --max-llm-calls 1

[agentic_hybrid] input=data/preprocessed/fincausal.jsonl
[agentic_hybrid] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.agentic_hybrid.vllm.jsonl
[agentic_hybrid] ontology_artifact=data/preprocessed/fincausal_olink_llm_embedding_onto_fibocoreplus.jsonl
[agentic_hybrid] kg_artifact=data/kg/fincausal__types=train.csv_skip=0_limit=None__kg.json
[agentic_hybrid] predict doc-types=all skip=0 limit=None
[agentic_hybrid] backend=vllm model=openai/gpt-oss-20b
[agentic_hybrid] few-shots collected: 3


AgenticHybrid on fincausal: 967doc [31:51,  1.98s/doc]

[agentic_hybrid] done. predicted=967
[agentic_hybrid] wrote: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.agentic_hybrid.vllm.jsonl


In [8]:
%run "scripts/eval_relations.py" \
  --dataset-key fincausal \
  --method agentic_hybrid \
  --backend vllm \
  --doc-type all

[eval] dataset-key: fincausal
[eval] method: agentic_hybrid
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/fincausal.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/fincausal.agentic_hybrid.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/fincausal/agentic_hybrid.vllm.all.json

=== Micro-level metrics ===
Precision: 0.9847
Recall:    0.9688
F1:        0.9767

=== Counts ===
TP: 900
FP: 14
FN: 29
num_docs_seen: 967
num_docs_eval: 967
num_docs_missing_gold: 0

[eval] Done.


### Maven Ere

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key maven_ere \
  --doc-types train \
  --skip 0

In [ ]:
try:
    %run "scripts/run_ontology_linking.py" --dataset-key maven_ere --ontology-key EventKG --method llm_embedding --doc-types all
except Exception as e:
    print(e)

#### Agent Single Hybrid RAG

In [9]:
%run "scripts/run_agentic_hybrid_relations.py" \
  --dataset-key maven_ere \
  --backend vllm \
  --doc-types all \
  --ontology-links-path data/preprocessed/maven_ere_olink_llm_embedding_onto_EventKG.jsonl \
  --ontology-key EventKG \
  --ontology-method llm_embedding \
  --kg-path data/kg/maven_ere__types=train_skip=0_limit=None__kg.json \
  --shot-num 3 \
  --shot-doc-types all \
  --max-llm-calls 1

[agentic_hybrid] input=data/preprocessed/maven_ere.jsonl
[agentic_hybrid] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere.agentic_hybrid.vllm.jsonl
[agentic_hybrid] ontology_artifact=data/preprocessed/maven_ere_olink_llm_embedding_onto_EventKG.jsonl
[agentic_hybrid] kg_artifact=data/kg/maven_ere__types=train_skip=0_limit=None__kg.json
[agentic_hybrid] predict doc-types=all skip=0 limit=None
[agentic_hybrid] backend=vllm model=openai/gpt-oss-20b
[agentic_hybrid] few-shots collected: 3


AgenticHybrid on maven_ere: 2870doc [35:41:24, 44.77s/doc]


HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:8000/v1/chat/completions

In [10]:
%run "scripts/eval_relations.py" \
  --dataset-key maven_ere \
  --method agentic_hybrid \
  --backend vllm \
  --doc-type all

[eval] dataset-key: maven_ere
[eval] method: agentic_hybrid
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/maven_ere.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/maven_ere.agentic_hybrid.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/maven_ere/agentic_hybrid.vllm.all.json

=== Micro-level metrics ===
Precision: 0.1328
Recall:    0.0817
F1:        0.1011

=== Counts ===
TP: 3018
FP: 19701
FN: 33940
num_docs_seen: 2870
num_docs_eval: 2870
num_docs_missing_gold: 0

[eval] Done.


### CausalBank

In [ ]:
try:
    %run "scripts/run_ontology_linking.py" --dataset-key causalbank --ontology-key wordnetfull --method llm_embedding --doc-types all
except Exception as e:
    print(e)

In [ ]:
%run "scripts/build_kg_from_preprocessed.py" \
  --dataset-key causalbank \
  --doc-types resulted_from \
  --skip 0

#### Agent Single Hybrid RAG

In [11]:
%run "scripts/run_agentic_hybrid_relations.py" \
  --dataset-key causalbank \
  --backend vllm \
  --doc-types all \
  --ontology-links-path data/preprocessed/causalbank_olink_llm_embedding_onto_wordnetfull.jsonl \
  --ontology-key wordnetfull \
  --ontology-method llm_embedding \
  --kg-path data/kg/causalbank__types=resulted_from_skip=0_limit=None__kg.json \
  --shot-num 3 \
  --shot-doc-types all \
  --max-llm-calls 1

[agentic_hybrid] input=data/preprocessed/causalbank.jsonl
[agentic_hybrid] output=/home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.agentic_hybrid.vllm.jsonl
[agentic_hybrid] ontology_artifact=data/preprocessed/causalbank_olink_llm_embedding_onto_wordnetfull.jsonl
[agentic_hybrid] kg_artifact=data/kg/causalbank__types=resulted_from_skip=0_limit=None__kg.json
[agentic_hybrid] predict doc-types=all skip=0 limit=None
[agentic_hybrid] backend=vllm model=openai/gpt-oss-20b
[agentic_hybrid] few-shots collected: 3


AgenticHybrid on causalbank: 1080doc [14:53:27, 49.64s/doc] 

[agentic_hybrid] done. predicted=1080
[agentic_hybrid] wrote: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.agentic_hybrid.vllm.jsonl


In [12]:
%run "scripts/eval_relations.py" \
  --dataset-key causalbank \
  --method agentic_hybrid \
  --backend vllm \
  --doc-type all

[eval] dataset-key: causalbank
[eval] method: agentic_hybrid
[eval] backend: vllm
[eval] doc-type: all
[eval] gold: data/preprocessed/causalbank.jsonl
[eval] preds: /home/galencarmedeiro/git/postdoc/ragtree/data/processed/causalbank.agentic_hybrid.vllm.jsonl
[eval] ignore-labels: ['null']
[eval] metrics-out: /home/galencarmedeiro/git/postdoc/ragtree/results/relations/causalbank/agentic_hybrid.vllm.all.json

=== Micro-level metrics ===
Precision: 0.7792
Recall:    0.3495
F1:        0.4826

=== Counts ===
TP: 58409
FP: 16553
FN: 108701
num_docs_seen: 1080
num_docs_eval: 1080
num_docs_missing_gold: 0

[eval] Done.
